In [1]:
import pandas as pd
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

# Enable progress_apply for Pandas
tqdm.pandas(desc="Geocoding Intersections")

print("1. Fetching FY2024 text-based camera tickets...")
url = (
    "https://data.cityofnewyork.us/resource/8zf9-spf8.csv?"
    "$select=street_name,intersecting_street,violation_code"
    "&$where=violation_code%20IN%20(7,%2036,%2071)"
    "&$limit=5000000"
)
fy2024_df = pd.read_csv(url)

print("2. Cleaning and reconstructing text locations...")
raw_text = fy2024_df['street_name'].fillna('') + fy2024_df['intersecting_street'].fillna('')

fy2024_df['Clean_Intersection'] = (
    raw_text
    .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace('@', ' AND ')
    .str.replace(r'^\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    + ', NYC'
)

print("3. Grouping to find unique camera locations...")
camera_counts = fy2024_df.groupby('Clean_Intersection').size().reset_index(name='Ticket_Count')
camera_counts = camera_counts.sort_values('Ticket_Count', ascending=False)

print("4. Initializing ArcGIS Geocoder...")
geolocator = ArcGIS()
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.5)

print(f"5. Geocoding all {len(camera_counts):,} locations...")
camera_counts['Location_Data'] = camera_counts['Clean_Intersection'].progress_apply(geocode)

camera_counts['Latitude'] = camera_counts['Location_Data'].apply(lambda loc: loc.latitude if loc else None)
camera_counts['Longitude'] = camera_counts['Location_Data'].apply(lambda loc: loc.longitude if loc else None)

final_camera_locations = camera_counts.drop(columns=['Location_Data']).dropna(subset=['Latitude', 'Longitude'])

output_path = '../data/nyc_camera_coordinates.csv'
final_camera_locations.to_csv(output_path, index=False)
print(f"\nSaved master locations to: {output_path}")

1. Fetching FY2024 text-based camera tickets...
2. Cleaning and reconstructing text locations...
3. Grouping to find unique camera locations...
4. Initializing ArcGIS Geocoder...
5. Geocoding all 2,650 locations...


Geocoding Intersections: 100%|██████████| 2650/2650 [24:34<00:00,  1.80it/s]


Saved master locations to: ../data/nyc_camera_coordinates.csv
